In [5]:
# %% [markdown]
# # DNA Methylation Prediction  
# **Proveniencia**  
# A jegyzetfüzet alapja a Kaggle-ról származó [DNA Methylation Prediction From Sequence](https://www.kaggle.com/code/gertthijs/dna-methylation-prediction-from-sequence), a többi rész saját munka.

# %% [code]
# ==== Importok és kezdeti beállítások ====================
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

# Csak a lokális három fájlra támaszkodunk:
print("Available CSV files:")
for f in ["train.csv","test.csv"]:
    print(" -", f)

# Konstansok
WINDOW_SIZE = 256
BATCH_SIZE  = 256

# %% [markdown]
# ## 1. Adatbetöltés & oszlopok ellenőrzése

# %% [code]
df_train = pd.read_csv('train.csv', index_col=0)
df_test  = pd.read_csv('test.csv',  index_col=0)
print("Train columns:", df_train.columns.tolist())
print("Test  columns:",  df_test.columns.tolist())

# %% [markdown]
# ## 2. Feature‐engineering: UCSC_RefGene_Group darabszám‐kódolás

# %% [code]
from collections import Counter

def encode_refgene_counts(df):
    s = df['UCSC_RefGene_Group'].fillna('')
    counters = s.apply(lambda text: Counter(text.split(';')))
    keys = sorted({k for cnt in counters for k in cnt if k})
    mat = np.array([[cnt.get(k, 0) for k in keys] for cnt in counters])
    cols = [f'rg_count_{k}' for k in keys]
    return pd.DataFrame(mat, columns=cols, index=df.index)

d_ref_train = encode_refgene_counts(df_train)
d_ref_test  = encode_refgene_counts(df_test)

# %% [markdown]
# ## 3. One‐hot DNS kódolás (szekvencia‐input)

# %% [code]
def one_hot_encoding(dna_seq):
    # A,C,G,T,N → 0..4
    nt2i = dict(zip('ACGTN', range(5)))
    nums = [nt2i[n] for n in dna_seq]
    oh   = np.eye(5)[nums]
    return oh[:, :4]  # csak A,C,G,T

def encode_all_sequences(df, window):
    seqs = df['seq'].apply(lambda s: one_hot_encoding(
        s[1000-window:1000+window]
    ))
    return np.stack(seqs.values, axis=0)  # (n_samples, 2*window, 4)

# %% [markdown]
# ## 4. Adatsplit & bemenet‐előkészítés

# %% [code]
# Tabuláris jellemzők és cél
X_tab_full = d_ref_train.values           # (n_samples, n_ref_features)
y_full     = df_train['Beta'].values      # (n_samples,)

# Szekvencia bemenet
X_seq_full = encode_all_sequences(df_train, WINDOW_SIZE)  # (n_samples,2*window,4)

# Egyetlen train/val split, utána reshapeeljük a y-t (n,1)-re
X_seq_tr, X_seq_va, X_tab_tr, X_tab_va, y_tr, y_va = train_test_split(
    X_seq_full, X_tab_full, y_full,
    test_size=0.2, random_state=42
)

# A rank-mismatch elkerülése miatt: labels legyen (batch,1)
y_tr = y_tr.reshape(-1, 1)
y_va = y_va.reshape(-1, 1)

print("Shapes after split:")
print("  X_seq_tr:", X_seq_tr.shape, "  X_tab_tr:", X_tab_tr.shape, "  y_tr:", y_tr.shape)
print("  X_seq_va:", X_seq_va.shape, "  X_tab_va:", X_tab_va.shape, "  y_va:", y_va.shape)

# %% [markdown]
# ## 5. EarlyStopping callback

# %% [code]
es_cb = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=5, restore_best_weights=True, start_from_epoch=10
)

# %% [markdown]
# ## 6. Modellek definíciói

# %% [code]
def build_orig(window):
    return tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(2*window,4)),
        tf.keras.layers.Conv1D(32, 4, activation='relu'),
        tf.keras.layers.AveragePooling1D(4,2),
        tf.keras.layers.Conv1D(64, 4, activation='relu'),
        tf.keras.layers.AveragePooling1D(4,2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ], name='orig_cnn')

def build_cnn_ext(window):
    return tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(2*window,4)),
        tf.keras.layers.Conv1D(64, 4, activation='relu'),
        tf.keras.layers.AveragePooling1D(4,2),
        tf.keras.layers.Conv1D(128,4, activation='relu'),
        tf.keras.layers.AveragePooling1D(4,2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ], name='cnn_ext')

def build_rnn_bi(window):
    return tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(2*window,4)),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ], name='rnn_bi')

def build_hybrid(window, tab_dim):
    seq_in = tf.keras.layers.Input(shape=(2*window,4), name='sequence_input')
    x = tf.keras.layers.Conv1D(64,4,activation='relu')(seq_in)
    x = tf.keras.layers.AveragePooling1D(4,2)(x)
    x = tf.keras.layers.Conv1D(128,4,activation='relu')(x)
    x = tf.keras.layers.AveragePooling1D(4,2)(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(64,activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    tab_in = tf.keras.layers.Input(shape=(tab_dim,), name='tabular_input')
    y = tf.keras.layers.Dense(32,activation='relu')(tab_in)
    y = tf.keras.layers.BatchNormalization()(y)
    y = tf.keras.layers.Dropout(0.3)(y)
    combined = tf.keras.layers.concatenate([x,y])
    z = tf.keras.layers.Dense(32,activation='relu')(combined)
    z = tf.keras.layers.BatchNormalization()(z)
    z = tf.keras.layers.Dropout(0.3)(z)
    out = tf.keras.layers.Dense(1, activation='sigmoid')(z)
    return tf.keras.Model([seq_in,tab_in], out, name='hybrid')

# %% [markdown]
# ## 7. Compile & Train minden modell

# %% [code]
METRICS = [
    tf.keras.metrics.BinaryCrossentropy(name='cross_entropy'),
    tf.keras.metrics.TruePositives(name='tp'),
    tf.keras.metrics.FalsePositives(name='fp'),
    tf.keras.metrics.TrueNegatives(name='tn'),
    tf.keras.metrics.FalseNegatives(name='fn'),
    tf.keras.metrics.BinaryAccuracy(name='accuracy'),
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall'),
    tf.keras.metrics.AUC(name='auc'),
    tf.keras.metrics.AUC(name='prc', curve='PR'),
]

models = {
    'orig_cnn': build_orig(WINDOW_SIZE),
    'cnn_ext':  build_cnn_ext(WINDOW_SIZE),
    'rnn_bi':   build_rnn_bi(WINDOW_SIZE),
    'hybrid':   build_hybrid(WINDOW_SIZE, X_tab_tr.shape[1]),
}

histories, trained = {}, {}
for name, model in models.items():
    print(f"\n--- Training {name} ---")
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=METRICS)
    if name == 'hybrid':
        hist = model.fit(
            x={'sequence_input': X_seq_tr, 'tabular_input': X_tab_tr},
            y=y_tr,
            validation_data=({'sequence_input': X_seq_va, 'tabular_input': X_tab_va}, y_va),
            epochs=50, batch_size=BATCH_SIZE, callbacks=[es_cb], verbose=2
        )
    else:
        hist = model.fit(
            x=X_seq_tr, y=y_tr,
            validation_data=(X_seq_va, y_va),
            epochs=50, batch_size=BATCH_SIZE, callbacks=[es_cb], verbose=2
        )
    histories[name] = hist
    trained[name]   = model

# %% [markdown]
# ## 8. Eredmények értékelése

# %% [code]
def eval_model(name, model, Xseq, Xtab, y, hybrid=False):
    if hybrid:
        y_score = model.predict({'sequence_input': Xseq, 'tabular_input': Xtab},
                                 batch_size=BATCH_SIZE)
    else:
        y_score = model.predict(Xseq, batch_size=BATCH_SIZE)
    y_pred = (y_score > 0.5).astype(int)
    acc   = accuracy_score(y, y_pred)
    prec  = precision_score(y, y_pred)
    rec   = recall_score(y, y_pred)
    f1    = f1_score(y, y_pred)
    try:
        auc = roc_auc_score(y, y_score)
        prc = average_precision_score(y, y_score)
    except ValueError:
        auc = prc = np.nan
    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    return {
        'Model': name, 'Accuracy': acc, 'Precision': prec,
        'Recall': rec, 'F1-score': f1, 'AUC': auc, 'PRC': prc,
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp
    }

rows = []
for m in ['orig_cnn','cnn_ext','rnn_bi']:
    rows.append(eval_model(f"{m} (train)", trained[m], X_seq_tr, None, y_tr))
    rows.append(eval_model(f"{m} (val)",   trained[m], X_seq_va, None, y_va))
rows.append(eval_model("hybrid (train)", trained['hybrid'], X_seq_tr, X_tab_tr, y_tr, hybrid=True))
rows.append(eval_model("hybrid (val)",   trained['hybrid'], X_seq_va, X_tab_va, y_va, hybrid=True))

df_results = pd.DataFrame(rows)
print(df_results)


Available CSV files:
 - train.csv
 - test.csv
Train columns: ['CHR', 'MAPINFO', 'UCSC_CpG_Islands_Name', 'UCSC_RefGene_Group', 'Relation_to_UCSC_CpG_Island', 'Regulatory_Feature_Group', 'Forward_Sequence', 'seq', 'Beta']
Test  columns: ['CHR', 'MAPINFO', 'UCSC_CpG_Islands_Name', 'UCSC_RefGene_Group', 'Relation_to_UCSC_CpG_Island', 'Regulatory_Feature_Group', 'Forward_Sequence', 'seq']
Shapes after split:
  X_seq_tr: (23252, 512, 4)   X_tab_tr: (23252, 6)   y_tr: (23252, 1)
  X_seq_va: (5813, 512, 4)   X_tab_va: (5813, 6)   y_va: (5813, 1)

--- Training orig_cnn ---
Epoch 1/50
91/91 - 5s - 55ms/step - accuracy: 0.7505 - auc: 0.8630 - cross_entropy: 0.5028 - fn: 1132.0000 - fp: 4670.0000 - loss: 0.5028 - prc: 0.7381 - precision: 0.5576 - recall: 0.8387 - tn: 11565.0000 - tp: 5885.0000 - val_accuracy: 0.6895 - val_auc: 0.9216 - val_cross_entropy: 0.6408 - val_fn: 1805.0000 - val_fp: 0.0000e+00 - val_loss: 0.6408 - val_prc: 0.8100 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_

/home/Viki/.conda/envs/py310_tf/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


91/91 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
              Model  Accuracy  Precision    Recall  F1-score       AUC  \
0  orig_cnn (train)  0.950241   0.875834  0.973065  0.921893  0.992243   
1    orig_cnn (val)  0.900396   0.789698  0.925762  0.852334  0.963658   
2   cnn_ext (train)  0.965938   0.903854  0.992732  0.946210  0.998176   
3     cnn_ext (val)  0.898847   0.790176  0.918006  0.849308  0.964179   
4    rnn_bi (train)  0.698177   0.000000  0.000000  0.000000  0.716125   
5      rnn_bi (val)  0.689489   0.000000  0.000000  0.000000  0.713956   
6    hybrid (train)  0.985378   0.979188  0.972210  0.975686  0.998265   
7      hybrid (val)  0.924480   0.877765  0.879224  0.878494  0.972772   

        PRC     TN   FP    FN    TP  
0  0.986858  15267  968   189  6828  
1  0.930901   3563  445   134  1671  
2  0.996057  15494  741    51  6966  
3  0.932394   3568  440   148  1657  
4  0.450825  16234    1  7017     0  
5  0.456749   4008    0  